# LSTM-GCN Training & Evaluation

This notebook trains an LSTM-GCN model for delta-neutral straddle momentum trading.

**Architecture**: Shared LSTM per ticker → Graph Convolutional Network → Position output

All hyperparameters and graph settings can be configured in Section 1 below.

## 0. Setup & Installation

In [ ]:
# Install dependencies (run on Colab)
import sys
if 'google.colab' in sys.modules:
    !pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral
    print("Dependencies installed!")

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Detect environment and set working directory
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/4YP-main'  # UPDATE THIS PATH
else:
    PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if not os.path.exists(os.path.join(PROJECT_DIR, 'gml')):
        PROJECT_DIR = '..'  

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Apply compatibility fixes for Keras 3 / NumPy 2.0
import re

def apply_fixes(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    
    original = content
    
    # Fix np.NINF -> -np.inf
    content = content.replace('np.NINF', '-np.inf')
    
    # Fix get_best_models -> load_model
    content = re.sub(
        r'best_model = self\.tuner\.get_best_models\(num_models=1\)\[0\]',
        'best_model = self.load_model(best_hp)',
        content
    )
    
    # Add restore_best_weights=True if missing
    if 'restore_best_weights=True' not in content:
        content = content.replace(
            'min_delta=1e-4,\n            )\n        ]\n        training_history = best_model.fit',
            'min_delta=1e-4,\n                restore_best_weights=True,\n            )\n        ]\n        training_history = best_model.fit'
        )
    
    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        print(f"Fixed: {filepath}")
    else:
        print(f"No changes needed: {filepath}")

apply_fixes('gml/graph_model_2.py')
apply_fixes('gml/deep_neural_network.py')

## 1. Configuration

**Modify the settings below to customize training.**

In [ ]:
#==============================================================================
# TRAINING CONFIGURATION - MODIFY THESE SETTINGS
#==============================================================================

# Experiment name (used for saving results)
EXPERIMENT_NAME = "lstm_gcn_experiment"

# Data split
TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023

# Feature file path
FEATURES_FILE = "data/straddle_features/features.csv"

#==============================================================================
# GRAPH CONFIGURATION
#==============================================================================

# Graph type: "cvx" or "pearson"
GRAPH_TYPE = "cvx"

# CVX graph parameters (only used if GRAPH_TYPE = "cvx")
CVX_ALPHA = 100
CVX_BETA = 0.01

# Pearson graph parameters (only used if GRAPH_TYPE = "pearson")
PEARSON_TAU = 0.45

#==============================================================================
# MODEL HYPERPARAMETERS
#==============================================================================

# LSTM settings
HIDDEN_LAYER_SIZE = 10      # LSTM hidden units
DROPOUT_RATE = 0.4          # Dropout rate
TIME_STEPS = 20             # Lookback window

# GCN settings
GCN_UNITS = 16              # GCN output dimension
NUM_GCN_LAYERS = 2          # Number of GCN layers (1 or 2)

# Training settings
LEARNING_RATE = 0.001       # Adam learning rate
MAX_GRADIENT_NORM = 0.01    # Gradient clipping
BATCH_SIZE = 32             # Mini-batch size
NUM_EPOCHS = 300            # Maximum epochs
EARLY_STOPPING_PATIENCE = 25  # Early stopping patience

# Hyperparameter search (set to 1 for single config, >1 for random search)
HP_SEARCH_ITERATIONS = 1

#==============================================================================
print("Configuration loaded!")
print(f"  Experiment: {EXPERIMENT_NAME}")
print(f"  Train: {TRAIN_START}-{TEST_START}, Test: {TEST_START}-{TEST_END}")
print(f"  Graph: {GRAPH_TYPE}" + (f" (alpha={CVX_ALPHA}, beta={CVX_BETA})" if GRAPH_TYPE == 'cvx' else f" (tau={PEARSON_TAU})"))
print(f"  LSTM: hidden={HIDDEN_LAYER_SIZE}, dropout={DROPOUT_RATE}")
print(f"  GCN: units={GCN_UNITS}, layers={NUM_GCN_LAYERS}")

In [ ]:
# Override hp_grid.py settings with notebook configuration
import settings.hp_grid as hp_grid

hp_grid.HP_HIDDEN_LAYER_SIZE_GRAPH = [HIDDEN_LAYER_SIZE]
hp_grid.HP_DROPOUT_RATE_GRAPH = [DROPOUT_RATE]
hp_grid.HP_LEARNING_RATE_GRAPH = [LEARNING_RATE]
hp_grid.HP_MAX_GRADIENT_NORM_GRAPH = [MAX_GRADIENT_NORM]
hp_grid.HP_MINIBATCH_SIZE_GRAPH = [BATCH_SIZE]
hp_grid.HP_GCN_UNITS = [GCN_UNITS]
hp_grid.HP_ALPHA = [CVX_ALPHA]
hp_grid.HP_BETA = [CVX_BETA]

# Override fixed_params
import settings.fixed_params as fixed_params
fixed_params.MODEL_PARAMS_GRAPH['total_time_steps'] = TIME_STEPS
fixed_params.MODEL_PARAMS_GRAPH['num_epochs'] = NUM_EPOCHS
fixed_params.MODEL_PARAMS_GRAPH['early_stopping_patience'] = EARLY_STOPPING_PATIENCE
fixed_params.MODEL_PARAMS_GRAPH['random_search_iterations'] = HP_SEARCH_ITERATIONS

print("Hyperparameters overridden!")

In [ ]:
# Configure graph file path based on selection
if GRAPH_TYPE == "cvx":
    GRAPH_FILE = f"data/graph_structure/cvx_opt/{CVX_ALPHA}_{CVX_BETA}_cvx.csv"
else:
    GRAPH_FILE = f"data/graph_structure/pearson/{PEARSON_TAU}.csv"

print(f"Graph file: {GRAPH_FILE}")

# Verify file exists
if not os.path.exists(GRAPH_FILE):
    print(f"WARNING: Graph file not found!")
else:
    print("Graph file found!")

## 2. Load Data & Prepare Features

In [ ]:
from settings.default import ALL_TICKERS
from settings.fixed_params import MODEL_PARAMS_GRAPH
from gml.graph_model_inputs import GraphModelFeatures

# Load raw data
raw_data = pd.read_csv(FEATURES_FILE, index_col=0, parse_dates=True)
raw_data["date"] = raw_data["date"].astype("datetime64[ns]")

print(f"Data shape: {raw_data.shape}")
print(f"Date range: {raw_data['date'].min()} to {raw_data['date'].max()}")
print(f"Tickers: {raw_data['ticker'].nunique()}")

In [ ]:
# Prepare model parameters
params = MODEL_PARAMS_GRAPH.copy()
params["architecture"] = "LSTM-GCN"

# Create model features
model_features = GraphModelFeatures(
    raw_data,
    params["total_time_steps"],
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    split_tickers_individually=params["split_tickers_individually"],
    train_valid_ratio=params["train_valid_ratio"],
    time_features=params["time_features"],
    lags=params["force_output_sharpe_length"] if params["force_output_sharpe_length"] else None,
)

print(f"\nNumber of tickers: {model_features.num_tickers}")
print(f"Train samples: {model_features.train['inputs'].shape}")
print(f"Valid samples: {model_features.valid['inputs'].shape}")
print(f"Test samples: {model_features.test_sliding['inputs'].shape}")

## 3. Initialize & Train Model

In [ ]:
# Patch the model to use our selected graph file
filepath = 'gml/graph_model_2.py'
with open(filepath, 'r') as f:
    content = f.read()

# Replace graph file path
import re
content = re.sub(
    r'graph_file = os\.path\.join\("data", "graph_structure", "cvx_opt", f"\{alpha\}_\{beta\}_cvx\.csv"\)',
    f'graph_file = "{GRAPH_FILE}"',
    content
)
content = re.sub(
    r'graph_file = os\.path\.join\("data", "graph_structure", "pearson", "[0-9.]+\.csv"\)',
    f'graph_file = "{GRAPH_FILE}"',
    content
)
# Also handle hardcoded paths
content = re.sub(
    r'graph_file = "data/graph_structure/[^"]+"',
    f'graph_file = "{GRAPH_FILE}"',
    content
)

with open(filepath, 'w') as f:
    f.write(content)

print(f"Graph file set to: {GRAPH_FILE}")

In [ ]:
# Need to reload the module after patching
import importlib
import gml.graph_model_2
importlib.reload(gml.graph_model_2)
from gml.graph_model_2 import GraphLSTMDeepMomentumNetwork

# Create output directory
output_dir = os.path.join("results", EXPERIMENT_NAME, f"{TEST_START}-{TEST_END}")
hp_directory = os.path.join(output_dir, "hp")
os.makedirs(output_dir, exist_ok=True)

# Initialize model
dmn = GraphLSTMDeepMomentumNetwork(
    EXPERIMENT_NAME,
    hp_directory,
    hp_minibatch_size=[BATCH_SIZE],
    num_tickers=model_features.num_tickers,
    **params,
    **model_features.input_params,
)

print(f"Model initialized!")
print(f"Output directory: {output_dir}")

In [ ]:
# Train model
print("Starting training...")
print(f"  Max epochs: {NUM_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Batch size: {BATCH_SIZE}")
print()

best_hp, best_model, training_history = dmn.hyperparameter_search(
    model_features.train,
    model_features.valid
)

print("\nTraining complete!")
print("\nBest hyperparameters:")
for k, v in best_hp.items():
    print(f"  {k}: {v}")

## 4. Visualize Training

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(training_history.history['loss'], label='Train Loss', linewidth=2)
if 'val_loss' in training_history.history:
    ax.plot(training_history.history['val_loss'], label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (Negative Sharpe)')
ax.set_title('LSTM-GCN Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'loss_plot.png'), dpi=150)
plt.show()

print(f"Epochs trained: {len(training_history.history['loss'])}")
print(f"Final train loss: {training_history.history['loss'][-1]:.4f}")
if 'val_loss' in training_history.history:
    print(f"Final val loss: {training_history.history['val_loss'][-1]:.4f}")

## 5. Evaluate on Test Set

In [ ]:
# Get predictions on test set
test_data = model_features.test_sliding
inputs = test_data['inputs']
outputs = test_data['outputs']
time_data = test_data['date']

positions = best_model.predict(inputs)

# Use only last timestep from each window (correct evaluation)
positions_last = positions[:, :, -1, 0]
outputs_last = outputs[:, :, -1, 0]
time_last = time_data[:, :, -1, 0]

captured = positions_last * outputs_last

# Aggregate by day
df = pd.DataFrame({
    'time': pd.to_datetime(time_last.flatten()),
    'captured': captured.flatten()
})
daily_returns = df.groupby('time')['captured'].mean()

print(f"Test days: {len(daily_returns)}")

In [ ]:
# Calculate performance metrics
annual_ret = daily_returns.mean() * 252
annual_vol = daily_returns.std() * np.sqrt(252)
sharpe = annual_ret / annual_vol
hit_rate = (daily_returns > 0).mean()

# Sortino ratio
downside = daily_returns[daily_returns < 0].std() * np.sqrt(252)
sortino = annual_ret / downside if downside > 0 else np.nan

# Max drawdown
cum_returns = (1 + daily_returns).cumprod()
rolling_max = cum_returns.expanding().max()
drawdown = (cum_returns - rolling_max) / rolling_max
max_dd = drawdown.min()

# Calmar ratio
calmar = annual_ret / abs(max_dd) if max_dd != 0 else np.nan

print("\n" + "="*50)
print("LSTM-GCN PERFORMANCE METRICS")
print("="*50)
print(f"{'Annual Return':20s}: {annual_ret*100:>10.2f}%")
print(f"{'Annual Volatility':20s}: {annual_vol*100:>10.2f}%")
print(f"{'Sharpe Ratio':20s}: {sharpe:>10.2f}")
print(f"{'Sortino Ratio':20s}: {sortino:>10.2f}")
print(f"{'Max Drawdown':20s}: {max_dd*100:>10.2f}%")
print(f"{'Calmar Ratio':20s}: {calmar:>10.2f}")
print(f"{'Hit Rate':20s}: {hit_rate*100:>10.1f}%")
print("="*50)
print(f"\n{'RESCALED TO 15% VOL':20s}: {annual_ret * (0.15 / annual_vol)*100:>10.2f}% return")

In [ ]:
# Position distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(positions_last.flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Position Distribution')
axes[0].grid(True, alpha=0.3)

# Add stats
stats_text = f"Mean: {positions_last.mean():.3f}\nStd: {positions_last.std():.3f}\nMin: {positions_last.min():.3f}\nMax: {positions_last.max():.3f}"
axes[0].text(0.95, 0.95, stats_text, transform=axes[0].transAxes, fontsize=10,
             verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Daily returns distribution
axes[1].hist(daily_returns.values * 100, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Daily Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Daily Returns Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'distributions.png'), dpi=150)
plt.show()

## 6. Cumulative Returns

In [ ]:
# Plot cumulative returns
cumulative_returns = (1 + daily_returns).cumprod() - 1

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(cumulative_returns.index, cumulative_returns.values * 100, linewidth=1.5, color='blue')
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.fill_between(cumulative_returns.index, 0, cumulative_returns.values * 100, 
                where=cumulative_returns.values >= 0, alpha=0.3, color='green')
ax.fill_between(cumulative_returns.index, 0, cumulative_returns.values * 100, 
                where=cumulative_returns.values < 0, alpha=0.3, color='red')

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (%)')
ax.set_title(f'LSTM-GCN Cumulative Returns ({TEST_START}-{TEST_END})')
ax.grid(True, alpha=0.3)

# Add final return annotation
final_return = cumulative_returns.iloc[-1] * 100
ax.annotate(f'Final: {final_return:.1f}%', 
            xy=(cumulative_returns.index[-1], final_return),
            xytext=(10, 0), textcoords='offset points',
            fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'cumulative_returns.png'), dpi=150)
plt.show()

## 7. Save Results

In [ ]:
# Save all results
results_df = pd.DataFrame({
    'date': daily_returns.index,
    'daily_return': daily_returns.values,
    'cumulative_return': cumulative_returns.values
})
results_df.to_csv(os.path.join(output_dir, 'daily_returns.csv'), index=False)

# Save metrics
metrics = {
    'annual_return': float(annual_ret),
    'annual_volatility': float(annual_vol),
    'sharpe_ratio': float(sharpe),
    'sortino_ratio': float(sortino),
    'max_drawdown': float(max_dd),
    'calmar_ratio': float(calmar),
    'hit_rate': float(hit_rate),
}
with open(os.path.join(output_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=4)

# Save hyperparameters
with open(os.path.join(output_dir, 'hyperparameters.json'), 'w') as f:
    json.dump(best_hp, f, indent=4)

# Save config
config = {
    'experiment_name': EXPERIMENT_NAME,
    'train_start': TRAIN_START,
    'test_start': TEST_START,
    'test_end': TEST_END,
    'graph_type': GRAPH_TYPE,
    'graph_file': GRAPH_FILE,
    'hidden_layer_size': HIDDEN_LAYER_SIZE,
    'dropout_rate': DROPOUT_RATE,
    'gcn_units': GCN_UNITS,
    'num_gcn_layers': NUM_GCN_LAYERS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
}
with open(os.path.join(output_dir, 'config.json'), 'w') as f:
    json.dump(config, f, indent=4)

print(f"Results saved to: {output_dir}")
print("\nFiles:")
for f in os.listdir(output_dir):
    print(f"  - {f}")